In [26]:
import pandas as pd
import numpy as np
import gc
from sklearn.metrics import r2_score

### Preprocessing
- Read imputed and encoding data
- Split data by `split` feature

In [2]:
file_path = "../Week3/model_data_knn_after_encoding.csv.gz"

meta_data = ["ClosePrice","SaleMonth","split", "CloseDate"
]

# Read only the column names first
all_columns = pd.read_csv(file_path, nrows=0).columns

feature_cols = [col for col in all_columns if col not in meta_data]

# Read feature columns directly as float32
feature_dtypes = {
    col: np.float32
    for col in feature_cols
}

df = pd.read_csv(file_path,dtype=feature_dtypes,parse_dates=["CloseDate"])

# Blank cells represent one-hot encoded zeros
df[feature_cols] = df[feature_cols].fillna(0)

print("Remaining NaN:",df[feature_cols].isna().sum().sum())

# Get train and test row indexes and sort by CloseDate
train_index = df.loc[df["split"] == "train", ["CloseDate"]].sort_values("CloseDate").index

test_index = df.loc[df["split"] == "test", ["CloseDate"]].sort_values("CloseDate").index

# Directly create contiguous NumPy arrays
x_train = np.ascontiguousarray(df.loc[train_index, feature_cols].to_numpy(dtype=np.float32, copy=False))

x_test = np.ascontiguousarray(df.loc[test_index, feature_cols].to_numpy(dtype=np.float32, copy=False))

y_train = df.loc[train_index,"ClosePrice"].to_numpy(dtype=np.float32)

y_test = df.loc[test_index, "ClosePrice"].to_numpy(dtype=np.float32)

print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("x_train dtype:", x_train.dtype)
print("y_train dtype:", y_train.dtype)

# Confirm there are no missing values
print("NaN in x_train:", np.isnan(x_train).sum())
print("NaN in x_test:", np.isnan(x_test).sum())

# Free the original large DataFrame
del df
gc.collect()

Remaining NaN: 0
x_train shape: (129813, 967)
x_test shape: (12024, 967)
x_train dtype: float32
y_train dtype: float32
NaN in x_train: 0
NaN in x_test: 0


0

### Create CV

In [3]:
from sklearn.model_selection import TimeSeriesSplit

time_cv = TimeSeriesSplit(n_splits=3)

### Decision Tree with hyperparameter tuning

In [23]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Define the parameter combinations
param_grid = {
    "max_depth": [20,30,40],
    "min_samples_leaf": [ 30, 40, 50]
}

# Create the base model
dt = DecisionTreeRegressor(random_state=42)

# Create the grid search
grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    scoring="r2",
    cv=time_cv,
    n_jobs=1,
    verbose=2
)

# Search for the best parameters using training data
grid_search.fit(x_train, y_train)

# Retrieve the best fitted model
best_dt = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation R²:", grid_search.best_score_)

# Evaluate the selected model
train_pred = best_dt.predict(x_train)
test_pred = best_dt.predict(x_test)

print("Training R²:", r2_score(y_train, train_pred))
print("Testing R²:", r2_score(y_test, test_pred))

Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] END ..................max_depth=20, min_samples_leaf=30; total time=   1.8s
[CV] END ..................max_depth=20, min_samples_leaf=30; total time=   4.0s
[CV] END ..................max_depth=20, min_samples_leaf=30; total time=   6.6s
[CV] END ..................max_depth=20, min_samples_leaf=40; total time=   1.9s
[CV] END ..................max_depth=20, min_samples_leaf=40; total time=   4.1s
[CV] END ..................max_depth=20, min_samples_leaf=40; total time=   6.4s
[CV] END ..................max_depth=20, min_samples_leaf=50; total time=   1.9s
[CV] END ..................max_depth=20, min_samples_leaf=50; total time=   4.2s
[CV] END ..................max_depth=20, min_samples_leaf=50; total time=   6.7s
[CV] END ..................max_depth=30, min_samples_leaf=30; total time=   1.8s
[CV] END ..................max_depth=30, min_samples_leaf=30; total time=   3.9s
[CV] END ..................max_depth=30, min_samp

### Random Forest

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

rf_structure_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=1
)

structure_params = {
    "max_depth": [15, 20, 25, 30, 35],
    "min_samples_leaf": [5, 10, 20, 40],
    "max_features": ["sqrt", 0.2, 0.4],
    "max_samples": [0.5, 0.7]
}


rf_coarse_search = RandomizedSearchCV(
    estimator=rf_structure_model,
    param_distributions=structure_params,
    n_iter=16,
    scoring="r2",
    cv=time_cv,
    random_state=42,
    n_jobs=1,
    pre_dispatch=1,
    verbose=2,
    error_score="raise"
)

rf_coarse_search.fit(x_train, y_train)

best_rf = rf_coarse_search.best_estimator_

print("Best coarse structure:")
print(rf_coarse_search.best_params_)

print("Best coarse CV R²:")
print(rf_coarse_search.best_score_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END max_depth=20, max_features=0.4, max_samples=0.7, min_samples_leaf=5; total time=  18.0s
[CV] END max_depth=20, max_features=0.4, max_samples=0.7, min_samples_leaf=5; total time=  42.7s
[CV] END max_depth=20, max_features=0.4, max_samples=0.7, min_samples_leaf=5; total time= 1.2min
[CV] END max_depth=20, max_features=0.4, max_samples=0.7, min_samples_leaf=40; total time=  16.2s
[CV] END max_depth=20, max_features=0.4, max_samples=0.7, min_samples_leaf=40; total time=  38.9s
[CV] END max_depth=20, max_features=0.4, max_samples=0.7, min_samples_leaf=40; total time= 1.1min
[CV] END max_depth=15, max_features=sqrt, max_samples=0.7, min_samples_leaf=5; total time=   2.7s
[CV] END max_depth=15, max_features=sqrt, max_samples=0.7, min_samples_leaf=5; total time=   6.1s
[CV] END max_depth=15, max_features=sqrt, max_samples=0.7, min_samples_leaf=5; total time=   9.8s
[CV] END max_depth=25, max_features=sqrt, max_samples=0.7, m

In [9]:
# Use the best structure from the previous search
best_structure = rf_coarse_search.best_params_

# Build a Random Forest with the selected structure
rf_tree_count_model = RandomForestRegressor(
    max_depth=best_structure["max_depth"],
    min_samples_leaf=best_structure["min_samples_leaf"],
    max_features=best_structure["max_features"],
    max_samples=best_structure["max_samples"],
    random_state=42,
    n_jobs=1
)

# Compare different numbers of trees
rf_tree_count_search = GridSearchCV(
    estimator=rf_tree_count_model,
    param_grid={
        "n_estimators": [50, 100, 150]
    },
    scoring="r2",
    cv=time_cv,
    n_jobs=1,
    pre_dispatch=1,
    verbose=2,
    error_score="raise"
)

# Run the search
rf_tree_count_search.fit(x_train, y_train)

print("Best number of trees:")
print(rf_tree_count_search.best_params_)

print("Final Random Forest CV R²:")
print(rf_tree_count_search.best_score_)

Fitting 3 folds for each of 3 candidates, totalling 9 fits
[CV] END ....................................n_estimators=50; total time=  19.8s
[CV] END ....................................n_estimators=50; total time=  44.0s
[CV] END ....................................n_estimators=50; total time= 1.2min
[CV] END ...................................n_estimators=100; total time=  36.9s
[CV] END ...................................n_estimators=100; total time= 1.5min
[CV] END ...................................n_estimators=100; total time= 2.4min
[CV] END ...................................n_estimators=150; total time=  55.8s
[CV] END ...................................n_estimators=150; total time= 2.2min
[CV] END ...................................n_estimators=150; total time= 3.7min
Best number of trees:
{'n_estimators': 150}
Final Random Forest CV R²:
0.7123974013491967


### Get RF Result

In [22]:
# Retrieve the final Random Forest selected by GridSearchCV
best_rf = rf_tree_count_search.best_estimator_

# Generate predictions
rf_train_pred = best_rf.predict(x_train)
rf_test_pred = best_rf.predict(x_test)

# Calculate R²
rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_test_r2 = r2_score(y_test, rf_test_pred)

# Display results
print("Best Random Forest parameters:")
print(best_rf.get_params())

print("Final CV R²:", rf_tree_count_search.best_score_)
print("Training R²:", rf_train_r2)
print("Testing R²:", rf_test_r2)

Best Random Forest parameters:
{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 20, 'max_features': 0.4, 'max_leaf_nodes': None, 'max_samples': 0.7, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 5, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 150, 'n_jobs': 1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
Final CV R²: 0.7123974013491967
Training R²: 0.8219704737999358
Testing R²: 0.516750288637237


## Model Comparison and Behavior

The Linear Regression baseline achieved a test R² of 0.4714. The tuned Decision Tree produced a slightly higher test R² of 0.4777, while the tuned Random Forest achieved the highest test R² of 0.5168.

### Decision Tree

**Strengths**
- Captures nonlinear relationships and interactions between property features.
- More interpretable than an ensemble model.
- Slightly outperformed the Linear Regression baseline.

**Weaknesses**
- A single tree can be unstable and sensitive to changes in the training data.
- The training R² was higher than the test R², indicating some overfitting.
- Its improvement over the baseline was relatively small.

### Random Forest

**Strengths**
- Produced the highest test R² among the models tested.
- Combines multiple trees to create more stable predictions.
- Captures nonlinear relationships and complex feature interactions.

**Weaknesses**
- Requires substantially more computation and memory than Linear Regression or a single Decision Tree.
- Is less interpretable than the other models.
- The difference between training and test R² indicates that some overfitting remains.

### Conclusion

The Random Forest was the strongest Week 5 model, improving test R² from 0.4714 for Linear Regression to 0.5168. However, its test performance remained lower than its cross-validation and training performance, suggesting that the most recent test month may be more difficult to predict. Future feature engineering and advanced models may improve generalization.

------

### Week 6 — Feature Engineering

• Example of sample features you can engineer: bed/bath ratio, age of property 
in years

• Adding more detailed geographic layer using school districts: build a more 
detailed regional feature by spatially joining each property’s coordinates against 
the CA School District Areas 2024-25 boundaries 
(https://data.ca.gov/dataset/california-school-district-areas-2024-
25/resource/7dfaf005-58eb-45db-93b1-7aff091b2172) 

• Re-train models with the updated feature set.

• Deliverable: Updated notebook + table comparing old vs new feature sets, 
including the school district layer.

In [27]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import geopandas as gpd

from scipy import sparse

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

### Loead Data

In [30]:
# Week 3 model-ready encoded data
encoded_path = Path(
    "../Week3/model_data_knn_after_encoding.csv.gz"
)

# Week 3 readable data before one-hot encoding
readable_path = Path(
    "../Week3/model_data_knn_before_encoding.csv"
)

# Change only this filename if your downloaded GeoJSON has a different name
boundary_path = Path(
    "../data/DistrictAreas2526.geojson"
)

for file_path in [encoded_path, readable_path, boundary_path]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path.resolve()}"
        )

print("All required files were found.")

All required files were found.


In [31]:
meta_columns = [
    "ClosePrice",
    "SaleMonth",
    "split",
    "CloseDate"
]

# Read column names first
all_columns = pd.read_csv(
    encoded_path,
    nrows=0
).columns.tolist()

base_feature_columns = [
    column
    for column in all_columns
    if column not in meta_columns
]

# Read encoded features as float32 to reduce memory usage
feature_dtypes = {
    column: np.float32
    for column in base_feature_columns
}

encoded_df = pd.read_csv(
    encoded_path,
    dtype=feature_dtypes,
    parse_dates=["CloseDate"]
)

# Blank cells in the encoded CSV represent one-hot encoded zeros
encoded_df[base_feature_columns] = (
    encoded_df[base_feature_columns]
    .fillna(0)
)

# Preserve row metadata for alignment validation
row_metadata = encoded_df[
    meta_columns
].copy()

# Preserve chronological order inside train and test sets
train_idx = (
    encoded_df.loc[
        encoded_df["split"].eq("train"),
        ["CloseDate"]
    ]
    .sort_values("CloseDate")
    .index
)

test_idx = (
    encoded_df.loc[
        encoded_df["split"].eq("test"),
        ["CloseDate"]
    ]
    .sort_values("CloseDate")
    .index
)

# Convert the original encoded features to sparse matrices
base_X_train = sparse.csr_matrix(
    encoded_df.loc[
        train_idx,
        base_feature_columns
    ].to_numpy(
        dtype=np.float32,
        copy=True
    )
)

base_X_test = sparse.csr_matrix(
    encoded_df.loc[
        test_idx,
        base_feature_columns
    ].to_numpy(
        dtype=np.float32,
        copy=True
    )
)

y_train_week6 = encoded_df.loc[
    train_idx,
    "ClosePrice"
].to_numpy(dtype=np.float32)

y_test_week6 = encoded_df.loc[
    test_idx,
    "ClosePrice"
].to_numpy(dtype=np.float32)

print("Original train shape:", base_X_train.shape)
print("Original test shape:", base_X_test.shape)
print("Train target shape:", y_train_week6.shape)
print("Test target shape:", y_test_week6.shape)
print(
    "Test month:",
    encoded_df.loc[test_idx, "SaleMonth"].unique()
)

# Free the large dense DataFrame
del encoded_df
gc.collect()

Original train shape: (129813, 967)
Original test shape: (12024, 967)
Train target shape: (129813,)
Test target shape: (12024,)
Test month: <ArrowStringArray>
['2026-05']
Length: 1, dtype: str


0

## Property-Level Feature Engineering

Two property-level features were created from existing variables.

`PropertyAge` is calculated using the property's construction year and the year in which the property was sold. This represents the age of the property at the time of the transaction.

`BedBathRatio` is calculated by dividing the number of bedrooms by the number of bathrooms. Bathroom values equal to zero are temporarily treated as missing to avoid division by zero.

In [32]:
feature_df = pd.read_csv(
    readable_path,
    parse_dates=["CloseDate"],
    low_memory=False
)

feature_df["_row_id"] = np.arange(
    len(feature_df)
)

# Validate that the readable and encoded files contain the same rows
if len(feature_df) != len(row_metadata):
    raise ValueError(
        "The readable and encoded datasets have different row counts."
    )

metadata_match = (
    feature_df["split"].astype(str).reset_index(drop=True)
    .equals(
        row_metadata["split"].astype(str).reset_index(drop=True)
    )
    and
    feature_df["CloseDate"].reset_index(drop=True)
    .equals(
        row_metadata["CloseDate"].reset_index(drop=True)
    )
    and
    np.allclose(
        feature_df["ClosePrice"].to_numpy(dtype=np.float64),
        row_metadata["ClosePrice"].to_numpy(dtype=np.float64)
    )
)

if not metadata_match:
    raise ValueError(
        "The readable and encoded datasets are not in the same row order."
    )

print("Dataset row alignment confirmed.")
print("Readable data shape:", feature_df.shape)

del row_metadata
gc.collect()

Dataset row alignment confirmed.
Readable data shape: (141837, 57)


0

# Feature Engineering

In [33]:
# Property age at the time of sale
feature_df["PropertyAge"] = (
    feature_df["CloseDate"].dt.year
    - feature_df["YearBuilt"]
)

# Treat impossible property ages as missing
feature_df.loc[
    ~feature_df["PropertyAge"].between(0, 200),
    "PropertyAge"
] = np.nan


# Bedrooms divided by bathrooms
feature_df["BedBathRatio"] = (
    feature_df["BedroomsTotal"]
    / feature_df[
        "BathroomsTotalInteger"
    ].replace(0, np.nan)
)

# Remove possible infinite results
feature_df["BedBathRatio"] = (
    feature_df["BedBathRatio"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    feature_df[
        [
            "CloseDate",
            "YearBuilt",
            "PropertyAge",
            "BedroomsTotal",
            "BathroomsTotalInteger",
            "BedBathRatio"
        ]
    ].head()
)

print(
    "\nMissing engineered values:"
)

print(
    feature_df[
        ["PropertyAge", "BedBathRatio"]
    ].isna().sum()
)

   CloseDate  YearBuilt  PropertyAge  BedroomsTotal  BathroomsTotalInteger  \
0 2025-05-29     2003.0         22.0            5.0                    4.0   
1 2025-05-19     1940.0         85.0            2.0                    2.0   
2 2025-05-27     1955.0         70.0            3.0                    1.0   
3 2025-05-30     1988.0         37.0            3.0                    2.0   
4 2025-05-30     1986.0         39.0            3.0                    3.0   

   BedBathRatio  
0          1.25  
1          1.00  
2          3.00  
3          1.50  
4          1.00  

Missing engineered values:
PropertyAge     18
BedBathRatio     0
dtype: int64


## Unified School District Mapping

The California School District Areas 2025–26 GeoJSON boundary file was loaded using GeoPandas. Following the project instructions, only records where `DistrictType == "Unified"` were retained.

Properties with reliable, originally observed latitude and longitude values were converted into geographic points. Coordinates previously filled through KNN imputation were excluded from district assignment because an approximate coordinate may result in an incorrect district classification near a district boundary.

A spatial join was then used to identify the Unified School District polygon containing each property. Properties without a reliable Unified School District match were labeled as `Unknown`.

In [34]:
districts = gpd.read_file(
    boundary_path
)

required_district_columns = {
    "DistrictType",
    "DistrictName",
    "geometry"
}

missing_district_columns = (
    required_district_columns
    - set(districts.columns)
)

if missing_district_columns:
    raise KeyError(
        "Missing required GeoJSON columns: "
        f"{sorted(missing_district_columns)}"
    )

unified_districts = districts.loc[
    districts["DistrictType"]
    .astype(str)
    .str.strip()
    .eq("Unified"),
    ["DistrictName", "geometry"]
].copy()

print(
    "All district polygons:",
    len(districts)
)

print(
    "Unified district polygons:",
    len(unified_districts)
)

print(
    "Unique Unified district names:",
    unified_districts["DistrictName"].nunique()
)

All district polygons: 936
Unified district polygons: 345
Unique Unified district names: 344


In [35]:
# Use only originally observed coordinates.
# A missing-indicator value of 0 means the original value was available.
valid_coordinates = (
    feature_df["Latitude_missing_ind"].eq(0)
    & feature_df["Longitude_missing_ind"].eq(0)
    & feature_df["Latitude"].between(32, 42)
    & feature_df["Longitude"].between(-125, -114)
)

property_points = gpd.GeoDataFrame(
    feature_df.loc[
        valid_coordinates,
        [
            "_row_id",
            "Latitude",
            "Longitude"
        ]
    ].copy(),
    geometry=gpd.points_from_xy(
        feature_df.loc[
            valid_coordinates,
            "Longitude"
        ],
        feature_df.loc[
            valid_coordinates,
            "Latitude"
        ]
    ),
    crs="EPSG:4326"
)

# Both datasets must use the same CRS
unified_districts = (
    unified_districts
    .to_crs(property_points.crs)
)

print(
    "All properties:",
    len(feature_df)
)

print(
    "Properties with reliable coordinates:",
    len(property_points)
)

print(
    "Properties excluded from spatial join:",
    len(feature_df) - len(property_points)
)

All properties: 141837
Properties with reliable coordinates: 141785
Properties excluded from spatial join: 52


In [ ]:
"""
    
    Spacial Join
    
"""
district_join = gpd.sjoin(
    property_points,
    unified_districts,
    how="left",
    predicate="within"
)

duplicate_matches = (
    district_join["_row_id"]
    .duplicated()
    .sum()
)

district_match_rate = (
    district_join["DistrictName"]
    .notna()
    .mean()
)

print(
    "Duplicate property matches:",
    duplicate_matches
)

print(
    f"District match rate: {district_match_rate:.2%}"
)

if duplicate_matches > 0:
    raise ValueError(
        "Some properties matched multiple Unified districts. "
        "Inspect the overlapping district polygons before continuing."
    )

Duplicate property matches: 0
District match rate: 75.87%


In [37]:
"""Add DistrictName into feature_df"""
district_mapping = (
    district_join
    .set_index("_row_id")["DistrictName"]
)

feature_df["DistrictName"] = (
    feature_df["_row_id"]
    .map(district_mapping)
    .fillna("Unknown")
)

print(
    "Missing DistrictName:",
    feature_df["DistrictName"]
    .isna()
    .sum()
)

print(
    "Unknown DistrictName count:",
    feature_df["DistrictName"]
    .eq("Unknown")
    .sum()
)

print(
    "Unknown DistrictName rate:",
    f"{feature_df['DistrictName'].eq('Unknown').mean():.2%}"
)

print(
    "\nMost common district values:"
)

print(
    feature_df["DistrictName"]
    .value_counts()
    .head(10)
)

Missing DistrictName: 0
Unknown DistrictName count: 34266
Unknown DistrictName rate: 24.16%

Most common district values:
DistrictName
Unknown                 34266
Los Angeles Unified     14191
San Diego Unified        3831
Desert Sands Unified     2609
Capistrano Unified       2593
Palm Springs Unified     2314
Oakland Unified          1850
Long Beach Unified       1758
Corona-Norco Unified     1757
Hemet Unified            1702
Name: count, dtype: int64


### School District Mapping Results

The spatial join produced no duplicate property matches. Among properties with reliable original coordinates, 75.86% were successfully matched to a Unified School District.

The remaining properties were labeled as `Unknown`. An unmatched property does not necessarily indicate an invalid coordinate. Because the boundary dataset was filtered to retain only Unified School Districts, properties located in areas served by separate elementary and high school districts may not fall within a Unified School District polygon.

In [38]:
enriched_output_path = Path(
    "../data/model_data_feature_engineered_with_district.csv.gz"
)

feature_df.drop(
    columns=["_row_id"],
    errors="ignore"
).to_csv(
    enriched_output_path,
    index=False,
    compression="gzip"
)

print(
    "Enriched dataset saved locally to:",
    enriched_output_path.resolve()
)

Enriched dataset saved locally to: D:\IDX_Exchange\IDX-Exchange-summer-2026\data\model_data_feature_engineered_with_district.csv.gz


## Preprocessing the New Features

The original 967 encoded features are retained without modification.

The two new numeric features are median-imputed and standardized. `DistrictName` is treated as a categorical feature and one-hot encoded. The new preprocessor is fitted only on the training set and then applied to the test set to avoid data leakage.

In [39]:
engineered_numeric_columns = [
    "PropertyAge",
    "BedBathRatio"
]

engineered_categorical_columns = [
    "DistrictName"
]

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

district_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=50,
                sparse_output=True,
                dtype=np.float32
            )
        )
    ]
)

new_feature_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            engineered_numeric_columns
        ),
        (
            "district",
            district_pipeline,
            engineered_categorical_columns
        )
    ],
    remainder="drop",
    sparse_threshold=1.0
)

new_feature_columns = (
    engineered_numeric_columns
    + engineered_categorical_columns
)

new_feature_train_df = feature_df.loc[
    train_idx,
    new_feature_columns
].copy()

new_feature_test_df = feature_df.loc[
    test_idx,
    new_feature_columns
].copy()

engineered_X_train = (
    new_feature_preprocessor
    .fit_transform(new_feature_train_df)
)

engineered_X_test = (
    new_feature_preprocessor
    .transform(new_feature_test_df)
)

engineered_X_train = sparse.csr_matrix(
    engineered_X_train,
    dtype=np.float32
)

engineered_X_test = sparse.csr_matrix(
    engineered_X_test,
    dtype=np.float32
)

new_feature_names = (
    new_feature_preprocessor
    .get_feature_names_out()
)

print(
    "New train feature shape:",
    engineered_X_train.shape
)

print(
    "New test feature shape:",
    engineered_X_test.shape
)

print(
    "Number of newly encoded columns:",
    len(new_feature_names)
)

print(
    "First new feature names:",
    new_feature_names[:10]
)

New train feature shape: (129813, 198)
New test feature shape: (12024, 198)
Number of newly encoded columns: 198
First new feature names: ['numeric__PropertyAge' 'numeric__BedBathRatio'
 'district__DistrictName_ABC Unified'
 'district__DistrictName_Acton-Agua Dulce Unified'
 'district__DistrictName_Alameda Unified'
 'district__DistrictName_Albany City Unified'
 'district__DistrictName_Alhambra Unified'
 'district__DistrictName_Alvord Unified'
 'district__DistrictName_Antioch Unified'
 'district__DistrictName_Apple Valley Unified']


## Updated Feature Matrix

The engineered numeric and district features are appended to the original encoded feature matrix.

The target values, training observations, test observations, and train/test dates remain unchanged. Therefore, differences in model performance can be attributed to the updated feature set rather than a different data split.

In [40]:
updated_X_train = sparse.hstack(
    [
        base_X_train,
        engineered_X_train
    ],
    format="csr"
).astype(
    np.float32,
    copy=False
)

updated_X_test = sparse.hstack(
    [
        base_X_test,
        engineered_X_test
    ],
    format="csr"
).astype(
    np.float32,
    copy=False
)

print(
    "Original train shape:",
    base_X_train.shape
)

print(
    "Updated train shape:",
    updated_X_train.shape
)

print(
    "Original test shape:",
    base_X_test.shape
)

print(
    "Updated test shape:",
    updated_X_test.shape
)

print(
    "Number of added columns:",
    updated_X_train.shape[1]
    - base_X_train.shape[1]
)

# Free matrices that are no longer needed separately
del base_X_train
del base_X_test
del engineered_X_train
del engineered_X_test
del new_feature_train_df
del new_feature_test_df

gc.collect()

Original train shape: (129813, 967)
Updated train shape: (129813, 1165)
Original test shape: (12024, 967)
Updated test shape: (12024, 1165)
Number of added columns: 198


77

In [ ]:
# Combine the original encoded feature names
# with the newly engineered feature names
updated_feature_names = (
    list(base_feature_columns)
    + list(new_feature_names)
)

# Confirm that the number of names matches the matrix columns
assert len(updated_feature_names) == updated_X_train.shape[1]
assert len(updated_feature_names) == updated_X_test.shape[1]


# Retrieve metadata in exactly the same chronological order
# used to construct updated_X_train and updated_X_test
train_metadata = (
    row_metadata
    .loc[
        train_idx,
        ["split", "CloseDate", "SaleMonth", "ClosePrice"]
    ]
    .reset_index(drop=True)
)

test_metadata = (
    row_metadata
    .loc[
        test_idx,
        ["split", "CloseDate", "SaleMonth", "ClosePrice"]
    ]
    .reset_index(drop=True)
)


# Verify that the targets remain aligned
assert np.allclose(
    train_metadata["ClosePrice"].to_numpy(dtype=np.float32),
    y_train_week6
)

assert np.allclose(
    test_metadata["ClosePrice"].to_numpy(dtype=np.float32),
    y_test_week6
)


# Convert the sparse matrices into sparse pandas DataFrames
train_feature_output = pd.DataFrame.sparse.from_spmatrix(
    updated_X_train,
    columns=updated_feature_names
)

test_feature_output = pd.DataFrame.sparse.from_spmatrix(
    updated_X_test,
    columns=updated_feature_names
)


# Add split, date, month, and target columns
train_output = pd.concat(
    [
        train_metadata,
        train_feature_output
    ],
    axis=1
)

test_output = pd.concat(
    [
        test_metadata,
        test_feature_output
    ],
    axis=1
)


# Combine training and testing rows into one file
week6_model_data = pd.concat(
    [
        train_output,
        test_output
    ],
    axis=0,
    ignore_index=True
)


# Save inside the repository's data folder
week6_output_path = Path(
    "../data/model_data_feature_engineered_after_encoding.csv.gz"
)

week6_output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

week6_model_data.to_csv(
    week6_output_path,
    index=False,
    compression="gzip"
)


print(
    "Final Week 6 model data saved to:",
    week6_output_path.resolve()
)

print(
    "Saved shape:",
    week6_model_data.shape
)

print(
    "Train rows:",
    (week6_model_data["split"] == "train").sum()
)

print(
    "Test rows:",
    (week6_model_data["split"] == "test").sum()

## Re-train Models with the Updated Feature Set

The Linear Regression, Decision Tree, and Random Forest models are re-trained using the updated feature matrix.

The Decision Tree and Random Forest use the best hyperparameters selected during Week 5. The models are not re-tuned in this section because the purpose of the experiment is to isolate the effect of the new features.

In [42]:
model_builders = {
    "Linear Regression": (
        lambda: LinearRegression()
    ),

    "Decision Tree": (
        lambda: DecisionTreeRegressor(
            max_depth=30,
            min_samples_leaf=40,
            random_state=42
        )
    ),

    "Random Forest": (
        lambda: RandomForestRegressor(
            n_estimators=150,
            max_depth=20,
            min_samples_leaf=5,
            max_features=0.4,
            max_samples=0.7,
            random_state=42,
            n_jobs=1
        )
    )
}

In [43]:
updated_results = []

for model_name, build_model in model_builders.items():

    print(f"Training {model_name}...")

    model = build_model()

    model.fit(
        updated_X_train,
        y_train_week6
    )

    train_predictions = model.predict(
        updated_X_train
    )

    test_predictions = model.predict(
        updated_X_test
    )

    updated_train_r2 = r2_score(
        y_train_week6,
        train_predictions
    )

    updated_test_r2 = r2_score(
        y_test_week6,
        test_predictions
    )

    updated_results.append(
        {
            "Model": model_name,
            "Updated Train R²": updated_train_r2,
            "Updated Test R²": updated_test_r2
        }
    )

    print(
        f"{model_name} train R²: "
        f"{updated_train_r2:.4f}"
    )

    print(
        f"{model_name} test R²: "
        f"{updated_test_r2:.4f}"
    )

    print("-" * 40)

    # Free the fitted model before training the next model
    del model
    del train_predictions
    del test_predictions

    gc.collect()

Training Linear Regression...
Linear Regression train R²: 0.6616
Linear Regression test R²: 0.4854
----------------------------------------
Training Decision Tree...
Decision Tree train R²: 0.6958
Decision Tree test R²: 0.4768
----------------------------------------
Training Random Forest...
Random Forest train R²: 0.8179
Random Forest test R²: 0.5154
----------------------------------------


## Original vs. Updated Feature Comparison

The following table compares the original Week 4–5 test results with the models re-trained using `PropertyAge`, `BedBathRatio`, and the spatially joined Unified School District feature.

In [44]:
original_test_scores = {
    "Linear Regression": 0.47141584763981925,
    "Decision Tree": 0.4776749488400225,
    "Random Forest": 0.516750288637237
}

comparison_df = pd.DataFrame(
    updated_results
)

comparison_df["Original Test R²"] = (
    comparison_df["Model"]
    .map(original_test_scores)
)

comparison_df["Test R² Change"] = (
    comparison_df["Updated Test R²"]
    - comparison_df["Original Test R²"]
)

comparison_df = comparison_df[
    [
        "Model",
        "Original Test R²",
        "Updated Train R²",
        "Updated Test R²",
        "Test R² Change"
    ]
]

display(
    comparison_df.round(4)
)

,Model,Original Test R²,Updated Train R²,Updated Test R²,Test R² Change
0,Linear Regression,0.4714,0.6616,0.4854,0.0140
1,Decision Tree,0.4777,0.6958,0.4768,-0.0009
2,Random Forest,0.5168,0.8179,0.5154,-0.0013


In [45]:
for _, row in comparison_df.iterrows():

    change = row["Test R² Change"]

    if change > 0:
        direction = "improved"
    elif change < 0:
        direction = "decreased"
    else:
        direction = "did not change"

    print(
        f"{row['Model']}: test R² {direction} "
        f"from {row['Original Test R²']:.4f} "
        f"to {row['Updated Test R²']:.4f} "
        f"({change:+.4f})."
    )

best_updated_row = comparison_df.loc[
    comparison_df["Updated Test R²"].idxmax()
]

print(
    "\nBest updated model:",
    best_updated_row["Model"]
)

print(
    "Best updated test R²:",
    f"{best_updated_row['Updated Test R²']:.4f}"
)

Linear Regression: test R² improved from 0.4714 to 0.4854 (+0.0140).
Decision Tree: test R² decreased from 0.4777 to 0.4768 (-0.0009).
Random Forest: test R² decreased from 0.5168 to 0.5154 (-0.0013).

Best updated model: Random Forest
Best updated test R²: 0.5154


## Week 6 Conclusion

The engineered feature set improved the Linear Regression test R² from
0.4714 to 0.4854, an increase of 0.0140. This suggests that explicitly
providing property age, the bedroom-to-bathroom ratio, and school district
information helped the linear model represent relationships that were not
fully captured by the original features.

The Decision Tree and Random Forest test R² values decreased slightly by
0.0009 and 0.0013, respectively. These changes are very small and do not
indicate a meaningful deterioration in performance. The tree-based models
were already able to capture nonlinear relationships among YearBuilt,
BedroomsTotal, and BathroomsTotalInteger.

In addition, the original feature set already contained latitude, longitude,
postal code, and high school district information. Therefore, the new
Unified School District feature may be partially redundant with the existing
geographic variables. One-hot encoding DistrictName also added many sparse
columns, while approximately 24% of properties were assigned to the Unknown
category because the boundary data was restricted to Unified School
Districts.

Overall, the engineered features benefited Linear Regression but did not
meaningfully change the performance of the tree-based models. Random Forest
remained the strongest model, with an updated test R² of 0.5154.